In [1]:
import sys, json, numpy as np
sys.path.insert(0, "../submission"); sys.path.insert(0, "..")
import model as M

m = M.Model(live_rating=False)                      # no API calls anywhere in this notebook
print(M.subject_name({"normalized_name": "", "subject_features_extra": "model_identifier=gpt-5-mini;harness=x"}))
print(M.text_key("hello"), M.subject_key({"b": 1, "a": 2}))
print(M.embed(["gpt-5-mini", "claude-haiku-5-5"]).shape)   # (2, 384), downloads MiniLM on first run


gpt-5-mini
2cf24dba5fb0a30e {"a": 2, "b": 1}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(2, 384)


In [2]:
z = np.zeros(3); y = np.ones(3)
for prec in (0.0, 0.5, 2.0, 8.0):
    print(prec, round(M._fit_offset(z, y, prec), 3))
print("one wrong of four:", round(M._fit_offset(np.zeros(4), np.array([1, 1, 1, 0.]), 2.0), 3))


0.0 7.202
0.5 1.293
2.0 0.549
8.0 0.171
one wrong of four: 0.334


In [3]:
m.load_ratings("../data/ratings.jsonl")
for f in ["ratings_haiku.jsonl", "ratings_haiku_dev_full.jsonl"]:
    m.load_ratings(f"D:/Github Repos/AIMS/AIMS-Competition/data/{f}")
print(len(m.ratings), "rated items")
r = next(iter(m.ratings.values())); print(r)
text = "What is the capital of France?"
print("unrated ->", m._rating_logit("zzqx not a real item 12345"))


22303 rated items
{'p': 0.4, 'domain': 'multimodal'}
unrated -> 0.0


In [4]:
m.net = [(np.zeros((385, 1), np.float32), np.zeros(1, np.float32))]   # prior logit = 0 everywhere
s = {"normalized_name": "test-model"}
items = [{"benchmark_id": "b", "item_content": f"item {i}"} for i in range(4)]
print("no labels:", m.predict_many([[s, it] for it in items]))
labeled = [([s, items[0]], 1), ([s, items[1]], 1), ([s, items[2]], 1)]
print("3 correct:", m.predict_many([[s, it] for it in items], labeled))
other = {"normalized_name": "other-model"}
labeled += [([other, items[3]], 0)]
print("other failed item 3:", m.predict_many([[s, it] for it in items], labeled))
ctx = {"labels_remaining": 5, "items_remaining": 50}
print("acquire p=0.5:", m.acquire(None, 0.5, None, ctx), " p=0.95:", m.acquire(None, 0.95, None, ctx),
      " must-take:", m.acquire(None, 0.95, None, {"labels_remaining": 5, "items_remaining": 5}))


no labels: [0.5, 0.5, 0.5, 0.5]
3 correct: [0.7972144125410223, 0.7972144125410223, 0.7972144125410223, 0.7238026829559329]
other failed item 3: [0.7766493377864105, 0.7766493377864105, 0.7766493377864105, 0.5253309794806967]
acquire p=0.5: True  p=0.95: False  must-take: True


In [5]:
from pathlib import Path
tmp = Path("../data/_tmp_model"); tmp.mkdir(exist_ok=True)
m.save(tmp); m2 = M.Model.load(tmp)
print(json.loads((tmp / "meta.json").read_text()))
print("same weights:", all(np.array_equal(a[0], b[0]) and np.array_equal(a[1], b[1]) for a, b in zip(m.net, m2.net)))


{'n_layers': 1, 'bench_strength': 2.0, 'subject_strength': 2.0, 'item_strength': 0.5, 'prior_scale': 0.5, 'rating_weight': {'knowledge': 1.0, 'reasoning_math': 0.8, 'coding': 0.6, 'legal_medical_finance': 0.5, 'agentic_tool_use': 0.0, 'multimodal': 0.0, 'default': 0.3}, 'live_rating': False, 'max_live_ratings': 60000, 'acq_uncertainty': 0.6}
same weights: True


In [6]:
from evaluate import evaluate, DEV, usable_benchmarks, load_benchmark, split_pairs, HierarchicalMean
import time

m = M.Model(live_rating=False).load_ratings("../data/ratings.jsonl")
missing = total = 0
for b in DEV:
    recs = load_benchmark(b)
    text = {r["item_id"]: (r["item"].get("item_content") or "") for r in recs}
    for _, (train_items, eval_items) in split_pairs(recs, 0).items():
        for i in list(eval_items[:100]) + list(train_items[:31]):
            total += 1; missing += M.text_key(text[i]) not in m.ratings
print(f"{len(m.ratings)} ratings loaded; {missing} of {total} protocol items unrated")


21888 ratings loaded; 154 of 104249 protocol items unrated


In [7]:
train = [b for b in usable_benchmarks() if b not in DEV]
t0 = time.time(); m.fit(train)
print(f"{len(train)} training benchmarks, trained in {time.time() - t0:.0f}s, {len(m.net)} layers")


147 training benchmarks, trained in 11s, 3 layers


In [8]:
t0 = time.time()
r = evaluate(m, DEV, train_pool=usable_benchmarks(), verbose=True)
print(f"\nrebuilt model: ALC={r['alc']:.4f}  {time.time() - t0:.0f}s")
print({b: round(v, 3) for b, v in r["per_budget"].items()})
h = evaluate(HierarchicalMean(), DEV, train_pool=usable_benchmarks(), verbose=False)
print(f"hierarchical mean: ALC={h['alc']:.4f}")


  hallusionbench               pairs= 253  ALC=0.2244
  ikp                          pairs= 170  ALC=0.1254
  multi_swebench               pairs=  82  ALC=0.1582
  mmlupro                      pairs=  48  ALC=0.2107
  editbench                    pairs=  44  ALC=0.2219
  swebench_live                pairs=  44  ALC=0.2331
  researchcodebench            pairs=  31  ALC=0.1958
  legalbench                   pairs=  31  ALC=0.2460
  tau2_bench                   pairs=  29  ALC=0.1836
  real_webagents               pairs=  26  ALC=0.2017
  mtbbench                     pairs=  25  ALC=0.2432
  workbench_revisited          pairs=  24  ALC=0.1759
  hle                          pairs=  12  ALC=0.2367
  B0=0.221  B1=0.207  B3=0.200  B7=0.190  B15=0.178  B31=0.165
  Brier ALC = 0.1937 over 819 pairs

rebuilt model: ALC=0.1937  575s
{0: 0.221, 1: 0.207, 3: 0.2, 7: 0.19, 15: 0.178, 31: 0.165}
hierarchical mean: ALC=0.2285


In [ ]:
import sys, json, time, importlib, numpy as np
sys.path.insert(0, "../submission"); sys.path.insert(0, "..")
import model as M; importlib.reload(M)
from evaluate import evaluate, DEV, usable_benchmarks, load_benchmark, split_pairs, HierarchicalMean

# 1. the file the platform will import: loads from weights.npz + meta.json, settings as expected
m = M.Model.load()
assert [W.shape for W, _ in m.net] == [(385, 256), (256, 64), (64, 1)], "unexpected network shape"
assert json.load(open("../submission/meta.json")) == {
    "n_layers": 3, "bench_strength": 2.0, "subject_strength": 2.0, "item_strength": 0.5, "prior_scale": 0.5,
    "rating_weight": {"knowledge": 1.0, "reasoning_math": 0.8, "coding": 0.6, "legal_medical_finance": 0.5,
                      "agentic_tool_use": 0.0, "multimodal": 0.0, "default": 0.3},
    "live_rating": True, "max_live_ratings": 60000}, "meta.json differs from the intended settings"
assert not hasattr(m, "acquire"), "acquisition code still present"
print("load OK: 3 layers, settings as intended, no acquisition code")

# 2. offline ratings cover the protocol
m.live_rating = False
m.load_ratings("../data/ratings.jsonl")
missing = total = 0
for b in DEV:
    recs = load_benchmark(b)
    text = {r["item_id"]: (r["item"].get("item_content") or "") for r in recs}
    for _, (train_items, eval_items) in split_pairs(recs, 0).items():
        for i in list(eval_items[:100]) + list(train_items[:31]):
            total += 1; missing += M.text_key(text[i]) not in m.ratings
print(f"ratings: {len(m.ratings)} loaded, {missing} of {total} protocol items unrated ({100*missing/total:.1f}%)")

# 3. the shift fit and a toy prediction behave
assert abs(M._fit_offset(np.zeros(3), np.ones(3), 2.0) - 0.549) < 0.01
s = {"normalized_name": "test-model"}; its = [{"benchmark_id": "b", "item_content": f"item {i}"} for i in range(3)]
p0 = m.predict([s, its[0]]); p1 = m.predict([s, its[0]], [([s, its[1]], 1), ([s, its[2]], 1)])
assert p1 > p0, "two correct labels should raise the prediction"
print(f"shifts OK: toy prediction {p0:.3f} -> {p1:.3f} after two correct labels")

# 4. the DEV protocol, leave-one-benchmark-out on the full pool (the number that matters; ~10 min)


load OK: 3 layers, settings as intended, no acquisition code
ratings: 21888 loaded, 154 of 104249 protocol items unrated (0.1%)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

shifts OK: toy prediction 0.555 -> 0.706 after two correct labels
  hallusionbench               pairs= 253  ALC=0.2244
  ikp                          pairs= 170  ALC=0.1253
  multi_swebench               pairs=  82  ALC=0.1580
  mmlupro                      pairs=  48  ALC=0.2106
  editbench                    pairs=  44  ALC=0.2219
  swebench_live                pairs=  44  ALC=0.2331
  researchcodebench            pairs=  31  ALC=0.1952


MemoryError: Unable to allocate 8.35 GiB for an array with shape (5823413, 385) and data type float32

In [1]:
import sys, time
sys.path.insert(0, "../submission"); sys.path.insert(0, "..")
import model as M
from evaluate import evaluate, DEV, usable_benchmarks
m = M.Model.load(); m.live_rating = False; m.load_ratings("../data/ratings.jsonl")
t0 = time.time()
r = evaluate(m, DEV, train_pool=usable_benchmarks(), verbose=True)
print(f"\nFINAL MODEL ALC = {r['alc']:.4f}  (expect ~0.1937)  {time.time()-t0:.0f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  hallusionbench               pairs= 253  ALC=0.2243
  ikp                          pairs= 170  ALC=0.1253
  multi_swebench               pairs=  82  ALC=0.1578
  mmlupro                      pairs=  48  ALC=0.2115
  editbench                    pairs=  44  ALC=0.2216
  swebench_live                pairs=  44  ALC=0.2330
  researchcodebench            pairs=  31  ALC=0.1954
  legalbench                   pairs=  31  ALC=0.2456
  tau2_bench                   pairs=  29  ALC=0.1839
  real_webagents               pairs=  26  ALC=0.2014
  mtbbench                     pairs=  25  ALC=0.2447
  workbench_revisited          pairs=  24  ALC=0.1752
  hle                          pairs=  12  ALC=0.2366
  B0=0.221  B1=0.207  B3=0.200  B7=0.190  B15=0.178  B31=0.165
  Brier ALC = 0.1936 over 819 pairs

FINAL MODEL ALC = 0.1936  (expect ~0.1937)  437s
